# Explicit 181L receptor audit and bounded BNZ redocking

Owning issues: DockingMT #4/#5/#33; writer defect #44. This notebook inspects the retained original six-run evidence and independently re-evaluates recovered poses. It does **not** rerun those searches or certify biological preparation. See `181l_receptor_workflow.md` for hypotheses and remaining provider #223/#323/#310 work.

In [1]:
import gzip
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyunitwizard as puw
import dockingmt as dmt

assert sys.version_info[:2] == (3, 14)
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'dockingmt/core').is_dir())
assert Path(dmt.__file__).resolve().parent == root / 'dockingmt'
sys.path.insert(0, str(root))
from devtools.qualify_chemical_templates import load_181l_benzene
receipt = json.loads((root / 'devguide/validation/data/181l_receptor/checkpoint_2026-10-06.json').read_text())
path = root / receipt['evidence']['path']
compressed = path.read_bytes()
original = gzip.decompress(compressed)
assert hashlib.sha256(compressed).hexdigest() == receipt['evidence']['gzip_sha256']
assert hashlib.sha256(original).hexdigest() == receipt['evidence']['json_sha256']
record = json.loads(original)
assert record['schema'] == 'dockingmt.181l_receptor_audit@1'
json.dumps(record, allow_nan=False)
for relative, expected in record['consumer_source_sha256'].items():
    assert hashlib.sha256((root / relative).read_bytes()).hexdigest() == expected
print(record['qualified_provider_commit'], record['python'].split()[0], 'original evidence verified')

5bd893c85fe8d211663b2b1f865f5f1d2c382a90 3.14.7 original evidence verified


In [2]:
audit = record['audit']
decisions = audit['decisions']
assert decisions['declared_residue_states'][30] == 'HIE'
assert decisions['pH'] is None
pd.DataFrame([{'decision': key, 'choice': decisions[key]} for key in (
    'n_terminal_state', 'c_terminal_state', 'protonation_basis', 'water_policy',
    'other_components_policy', 'receptor_refinement', 'generated_H_name_policy')])

,decision,choice
0,n_terminal_state,ammonium
1,c_terminal_state,carboxylate
2,protonation_basis,caller_declared_hypothesis_without_environment...
3,water_policy,exclude_all_observed_waters
4,other_components_policy,exclude_all_nonprotein_components_except_BNZ_p...
5,receptor_refinement,none
6,generated_H_name_policy,explicit H plus three uppercase hexadecimal at...


In [3]:
mapping = decisions['source_to_repaired_atom_indices']
observed = np.asarray(audit['original_receptor_snapshot']['structures']['coordinates'])
expanded = np.asarray(audit['expanded_receptor_snapshot']['structures']['coordinates'])
np.testing.assert_allclose(expanded[:, mapping], observed, rtol=0, atol=1e-12)
assert len(mapping) == 1289 and len(set(mapping)) == 1289
assert decisions['repaired_to_full_source_atom_indices'].count(None) == 1
assert len(decisions['generated_H_name_map']) == 1313
assert all(len(row['name']) == 4 for row in decisions['generated_H_name_map'])
assert audit['original_source_unchanged']
charge = audit['receptor_charge_audit']
assert abs(charge['total_charge'] - 8) < 1e-8
assert charge['assessment'] == 'consistent'
pd.DataFrame([
    {'stage': 'observed heavy receptor', 'atoms': len(mapping), 'total_charge_e': None},
    {'stage': 'repaired heavy receptor', 'atoms': 1290, 'total_charge_e': None},
    {'stage': 'expanded evaluated receptor', 'atoms': 2603, 'total_charge_e': charge['partial_charge_assignment']['total_charge']},
    {'stage': 'projected receptor', 'atoms': audit['receptor_preparation']['n_atoms'], 'total_charge_e': charge['total_charge']},
    {'stage': 'rounded PDBQT', 'atoms': audit['receptor_preparation']['n_atoms'], 'total_charge_e': charge['pdbqt']['total_charge']},
])

,stage,atoms,total_charge_e
0,observed heavy receptor,1289,NaN
1,repaired heavy receptor,1290,NaN
2,expanded evaluated receptor,2603,8.000
3,projected receptor,1615,8.000
4,rounded PDBQT,1615,7.943


In [4]:
rows = []
for run in record['redocking_runs']:
    parameters = run['result']['protocol_info']['parameters']
    evaluation = run['evaluation']
    assert not parameters['allow_provisional_preparation']
    dmt.verify_captured_inputs(run['result']['provenance']['backend_artifacts'])
    assert run['result']['problem_info']['metadata']['preparation_decisions'] == decisions
    rows.append({'exhaustiveness': parameters['exhaustiveness'], 'seed': parameters['seed'],
                 'returned_poses': evaluation['n_poses'],
                 'first_RMSD_angstrom': evaluation['first_pose']['rmsd'],
                 'closest_RMSD_angstrom': evaluation['closest_pose']['rmsd'],
                 'first_score_kcal_per_mol': evaluation['first_pose']['scores']['vina'],
                 'top1_recovered': evaluation['top_n'][0]['recovered'],
                 'top5_recovered': evaluation['top_n'][1]['recovered']})
assert len(rows) == 6
for role in ('receptor', 'partner'):
    assert len({run['result']['provenance']['backend_artifacts'][role]['sha256']
                for run in record['redocking_runs']}) == 1
pd.DataFrame(rows)

,exhaustiveness,seed,returned_poses,first_RMSD_angstrom,closest_RMSD_angstrom,first_score_kcal_per_mol,top1_recovered,top5_recovered
0,1,7,2,2.107827,2.072376,-5.474,True,True
1,1,42,1,2.291607,2.291607,-5.482,True,True
2,1,2026,3,2.714441,2.353699,-5.482,False,True
3,8,7,4,2.285585,2.002621,-5.487,True,True
4,8,42,5,2.101558,1.980593,-5.494,True,True
5,8,2026,5,2.109056,1.937150,-5.484,True,True


Five of six first poses satisfy the declared 2.5 angstrom cutoff; all six return a recovered pose. This single rigid symmetric ligand is not a benchmark population. Source-identity positional RMSD has no symmetry correction or alignment. Hypothesis alternatives, environmental H refinement and scoring validity remain unassessed.

In [5]:
reference, _, _ = load_181l_benzene()
first = record['redocking_runs'][0]
restored = dmt.DockingResult.from_dict(first['result'])
with puw.context(standard_units=['pm', 'fs', 'K', 'mole', 'dalton', 'e', 'kJ/mol', 'radians']):
    repeated = dmt.evaluate_redocking(restored, reference, rmsd_cutoff=puw.quantity(250, 'pm'))
assert repeated['criterion'] == first['evaluation']['criterion']
for actual, expected in zip(repeated['poses'], first['evaluation']['poses']):
    np.testing.assert_allclose(actual['rmsd'], expected['rmsd'], rtol=0, atol=1e-12)
    assert {k: v for k, v in actual.items() if k != 'rmsd'} == {k: v for k, v in expected.items() if k != 'rmsd'}
assert all(restored.provenance['preparation'][role]['assessment'] == 'unassessed'
           for role in ('receptor', 'partner'))
print('Recovered JSON result independently re-evaluated under pm/fs; chemistry remains unassessed.')

Recovered JSON result independently re-evaluated under pm/fs; chemistry remains unassessed.


In [6]:
pd.DataFrame(record['warnings'])
print('Limits retained:', len(record['limits']))
record['limits']

Limits retained: 10


['One explicit receptor hypothesis, not predicted protonation or a biological preparation certificate.',
 'HIE, charged termini and excluded waters are recorded choices; their scientific suitability is unassessed.',
 'OXT and H geometry are generated by declared provider tools without environmental refinement (molsysmt#323).',
 'Gasteiger charges and chemical_environment@1 types identify executed models, not universal scoring validity.',
 'Vina parsing and near-native recovery do not independently validate chemical preparation or affinity.',
 'Six search controls on one rigid symmetric ligand are not a benchmark population.',
 'RMSD uses source identities without symmetry correction (molsysmt#310) or alignment.',
 'Temporary consumer projection/export remains under molsysmt#223; no AD4, H5MSM mechanics or public artifact qualification.',
 'Python 3.14 source evidence only; other Python minors, clean installed artifacts and hosted CI remain outstanding.',
 'Native artifact digest does n